[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 02](README.md)

# Sincronización, condición y deadlock

**Tema:** 02 · **Sesiones:** 8, 10 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué invariante protege cada primitiva y cómo se detecta un ciclo de espera?


## Resultados de aprendizaje

- Relacionar mutex, condición y barrera con invariantes.
- Explicar happens-before sin usar tiempo como sincronización.
- Detectar un ciclo en un grafo wait-for.


## Modelo conceptual

Un mutex protege un invariante, no una línea aislada.

Una variable de condición se espera dentro de un bucle que reevalúa el predicado.

Un deadlock requiere exclusión, retención y espera, no expropiación y espera circular.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "02"
NOTEBOOK = "02_memoria_compartida/02_sincronizacion.ipynb"
assert (ROOT / "curso" / "notebooks" / "02_memoria_compartida" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Grafo de espera

Se detecta un ciclo mediante búsqueda en profundidad.


In [ ]:
def has_cycle(graph):
    visiting, done = set(), set()
    def visit(node):
        if node in visiting: return True
        if node in done: return False
        visiting.add(node)
        if any(visit(next_node) for next_node in graph.get(node, ())): return True
        visiting.remove(node); done.add(node); return False
    return any(visit(node) for node in graph)
safe = {"T0": ["T1"], "T1": []}
deadlock = {"T0": ["T1"], "T1": ["T0"]}
assert not has_cycle(safe) and has_cycle(deadlock)
print("seguro:", has_cycle(safe), "deadlock:", has_cycle(deadlock))


**Interpretación.** Ordenar globalmente la adquisición de recursos rompe la condición de espera circular.


## Buffer acotado

Se comprueban invariantes de ocupación para una secuencia productor-consumidor.


In [ ]:
from collections import deque
capacity, buffer = 3, deque()
operations = [("put", 4), ("put", 7), ("get", None), ("put", 9), ("get", None), ("get", None)]
consumed = []
for operation, value in operations:
    if operation == "put":
        assert len(buffer) < capacity
        buffer.append(value)
    else:
        assert buffer
        consumed.append(buffer.popleft())
    assert 0 <= len(buffer) <= capacity
assert consumed == [4, 7, 9]
print(consumed)


**Interpretación.** En Pthreads, el predicado sería `count>0` o `count<capacity` protegido por el mismo mutex.


## Práctica reproducible

1. Anotar el invariante al lado de cada estado compartido.
2. Construir un caso que fuerce intercalaciones distintas.
3. Ejecutar ThreadSanitizer cuando el toolchain lo soporte.


## Errores frecuentes

- Usar `sleep` para ordenar hilos.
- Esperar condición con `if` en lugar de `while`.
- Bloquear recursos en órdenes diferentes.

## Criterios de aceptación

- Invariantes escritos y comprobados.
- Ausencia de ciclos en el orden de locks.
- Pruebas repetidas y detector de carreras documentado.


## Referencias y material relacionado

- [Mutex](../../../pthreads/thread_mutex.c)
- [Deadlock](../../../pthreads/thread_deadlock.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 02](README.md)
